In [21]:
import pandas as pd
import numpy as np

In [22]:
listing_file = "RentingOutofFlatsfromJan2021.csv"
listing_df = pd.read_csv(listing_file, index_col=None)
listing_df

,rent_approval_date,town,block,street_name,flat_type,monthly_rent
0,2021-01,ANG MO KIO,105,ANG MO KIO AVE 4,4-ROOM,2000
1,2021-01,ANG MO KIO,107,ANG MO KIO AVE 4,3-ROOM,1750
2,2021-01,ANG MO KIO,108,ANG MO KIO AVE 4,3-ROOM,1750
3,2021-01,ANG MO KIO,111,ANG MO KIO AVE 4,5-ROOM,2230
4,2021-01,ANG MO KIO,111,ANG MO KIO AVE 4,5-ROOM,2450
...,...,...,...,...,...,...
190838,2026-02,TAMPINES,150,TAMPINES ST 12,4-ROOM,3000
190839,2026-02,PASIR RIS,725,PASIR RIS ST 72,5-ROOM,3500
190840,2026-02,HOUGANG,1,HOUGANG AVE 3,3-ROOM,2500
190841,2026-02,BEDOK,720,BEDOK RESERVOIR RD,4-ROOM,3700


In [23]:
listing_df['available_from'] = pd.to_datetime(listing_df['rent_approval_date'], format="%Y-%m")
# keep only 2026
listing_2026 = listing_df[listing_df['available_from'].dt.year == 2026].copy()
print(f"Rows in 2026 dataset: {len(listing_2026)}")

Rows in 2026 dataset: 5890


In [24]:
listing_2026["title"] = listing_2026["flat_type"] + " Flat at " + listing_2026["street_name"]
listing_2026["address"] = 'Blk ' + listing_2026["block"] + ' ' + listing_2026["street_name"]
lease_options = [6,12,24]
listing_2026["lease"] = np.random.choice(lease_options, size = len(listing_2026))
rand_days = np.random.randint(0, 60, size=len(listing_2026))
listing_2026["available_from"] = listing_2026["available_from"] + pd.to_timedelta(rand_days, unit="D")

listing_2026 = listing_2026[[
    "title", "address", "monthly_rent", "lease", "flat_type", "available_from"
]]
print(listing_2026.head())

                                     title                      address  \
184915      3-ROOM Flat at UPP ALJUNIED RD     Blk 233C UPP ALJUNIED RD   
184916  4-ROOM Flat at CHOA CHU KANG AVE 1  Blk 202 CHOA CHU KANG AVE 1   
184917   5-ROOM Flat at BT PANJANG RING RD   Blk 619 BT PANJANG RING RD   
184918        5-ROOM Flat at ANCHORVALE RD       Blk 317C ANCHORVALE RD   
184919            4-ROOM Flat at TOWNER RD           Blk 149A TOWNER RD   

        monthly_rent  lease flat_type available_from  
184915          3500     24    3-ROOM     2026-01-25  
184916          3200     24    4-ROOM     2026-01-13  
184917          3000     24    5-ROOM     2026-02-17  
184918          3200     24    5-ROOM     2026-02-09  
184919          4000      6    4-ROOM     2026-02-11  


In [25]:
listing_2026["available_from"] = listing_2026["available_from"].dt.strftime("%Y-%m-%d")
with open("insert_listings.sql", "w") as f:
    for row in listing_2026.itertuples(index=False):
        f.write(
            "INSERT INTO Listing "
            "(title, address, rent, lease, flat_type, available_from) "
            f"VALUES {tuple(row)};\n"
        )